# 16-bit 精度 LoRA 微調實戰：以 ChatGLM3-6B 為例

## 學習目標

完成本 notebook 後，你將能夠：

1. 說明 bf16 / fp16 / fp32 三種精度的差異，以及為何 bf16 是 2026 年的預設選擇
2. 使用 2026 統一慣例（`device_map='auto'` + `torch_dtype=torch.bfloat16` + `use_safetensors=True`）載入大型語言模型
3. 設定 `LoraConfig` 並理解 rank、alpha、dropout 等超參數的意義
4. 使用 `trl.SFTTrainer` + `apply_chat_template()` 完成指令微調
5. 理解 `SFTTrainer` 底層的 response-only 標籤遮罩機制

## 與相鄰 notebook 的銜接

- **前置**：`../01-4bits_training/chatglm3_lora_4bit.ipynb` — 4-bit 量化訓練；本 notebook 改用 full 16-bit，不加 BitsAndBytesConfig
- **後續**：`../../05-Multimodal/` — 多模態微調；本 notebook 建立的 apply_chat_template 慣例是關鍵橋樑

## 硬體需求

| 精度 | 估計 VRAM | 備註 |
|------|-----------|------|
| bf16 full weights (6B) | ~14 GB | 本 notebook 預設路徑 |
| + LoRA adapter (r=8) | +~100 MB | 可忽略 |
| 訓練時 activation (bs=2, seq=256) | +~2-4 GB | 視序列長度 |

**VRAM 不足 16 GB？** 請改用 `../01-4bits_training/chatglm3_lora_4bit.ipynb`（4-bit 量化約需 6-8 GB）。

In [ ]:
# 版本鎖定：確保全 repo 環境一致
# 在 terminal 執行：pip install -r requirements.txt
# 或直接執行此 cell 安裝
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "trl>=0.12" \
    "peft>=0.13" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "torch>=2.4" \
    "evaluate>=0.4"

## Step 1：匯入套件

2026 寫法使用 `trl.SFTTrainer` + `SFTConfig`，不需要手動管理標籤遮罩。

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

set_seed(42)  # reproducibility: 固定所有隨機種子

## Step 2：載入資料集

2026 寫法優先使用 HuggingFace Hub 的 dataset id，做到零硬路徑依賴。

> **本 notebook 使用** `silk-road/alpaca-data-gpt4-chinese`，
> 等價於原版 `alpaca_data_zh` 的繁/簡中文指令資料集。

In [ ]:
# 從 HF Hub 載入，避免硬碟路徑依賴
# 若需使用本機資料：Dataset.load_from_disk("path/to/alpaca_data_zh")
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
print(ds)
ds[:3]

## Step 3：載入 Tokenizer

### 為何不再用 `d:/Pretrained_models/` 這類硬路徑？

- 硬路徑與作業系統、機器綁定，無法在 Colab / 伺服器 / CI 環境重現
- HuggingFace Hub model id（如 `THUDM/chatglm3-6b`）可自動快取、自動下載，且是跨環境的單一真相來源
- 若企業環境需要離線推論，可設 `TRANSFORMERS_CACHE` 環境變數指向本機快取目錄

In [ ]:
# 2026 統一慣例：使用 HF Hub model id
MODEL_ID = "THUDM/chatglm3-6b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,  # ChatGLM 系列需要此旗標載入自訂程式碼
)
print(tokenizer)
print("EOS token:", tokenizer.eos_token, "| EOS id:", tokenizer.eos_token_id)
print("Chat template 是否存在:", tokenizer.chat_template is not None)

## Step 4：資料集預處理

### `apply_chat_template()` 的設計語意

`tokenizer.apply_chat_template()` 是 transformers 4.34+ 引入的跨模型統一抽象：
- **訓練側**：`tokenize=False` → 得到帶有特殊標記的純文字字串，再統一 tokenize
- **推論側**：`add_generation_prompt=True` → 在字串尾端加上助理開始標記
- **多模態**：相同 API 的 `images=` / `videos=` 參數支援多模態訊息（05-Multimodal 模組的基礎）

下方展示底層手刻版（教學用），說明 -100 遮罩邏輯；生產路徑請直接使用 SFTTrainer（見 Step 6）。

In [ ]:
def process_func(example):
    """
    手刻版預處理（教學用）：示範 apply_chat_template + -100 遮罩邏輯。
    生產路徑請改用 SFTTrainer + formatting_func（見 Step 6）。
    """
    MAX_LENGTH = 256

    # --- 組裝 messages 結構（跨模型通用格式）---
    instruction_text = "\n".join(
        [example["instruction"], example["input"]]
    ).strip()

    messages = [
        {"role": "user", "content": instruction_text},
    ]

    # apply_chat_template: 輸出帶特殊標記的純文字（tokenize=False）
    # 與推論側共用同一路徑，確保訓練/推論 distribution 一致
    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 在尾端加 <|assistant|>\n，告知模型開始生成
    )

    # tokenize instruction（不含 eos）
    instruction_enc = tokenizer(prompt_str, add_special_tokens=False)

    # tokenize response（加 eos）
    response_enc = tokenizer(
        example["output"] + tokenizer.eos_token,
        add_special_tokens=False,
    )

    # 拼接：instruction + response
    input_ids = instruction_enc["input_ids"] + response_enc["input_ids"]
    attention_mask = instruction_enc["attention_mask"] + response_enc["attention_mask"]

    # -100 遮罩：只對 response 部分計算 loss
    # WHY: CrossEntropyLoss 遇到 label=-100 時自動跳過該 token
    # 若連 instruction 一起算 loss，模型會學到「背誦 prompt」，而非「根據 prompt 生成回答」
    labels = [-100] * len(instruction_enc["input_ids"]) + response_enc["input_ids"]

    # 截斷至 MAX_LENGTH
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# batched=False 保持原版對應，確認邏輯正確後再 batched=True 加速
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
print(tokenized_ds)

In [ ]:
# 驗證 1：解碼完整 input_ids，確認 prompt + response 格式正確
print("=== input_ids decoded ===")
print(tokenizer.decode(tokenized_ds[1]["input_ids"]))

In [ ]:
# 驗證 2：只解碼 labels 中非 -100 的部分（即 response tokens），確認遮罩正確
print("=== response labels decoded ===")
response_tokens = [x for x in tokenized_ds[1]["labels"] if x != -100]
print(tokenizer.decode(response_tokens))

## Step 5：載入模型

### bf16 vs fp16 vs fp32

| 格式 | 符號位 | 指數位 | 尾數位 | 動態範圍 | 精度 | 備註 |
|------|--------|--------|--------|----------|------|------|
| fp32 | 1 | 8 | 23 | 大 | 高 | 訓練傳統預設，VRAM 耗用最大 |
| fp16 | 1 | 5 | 10 | **小** | 中 | 容易 NaN/overflow，需 loss scaling |
| bf16 | 1 | **8** | 7 | **與 fp32 相同** | 較低 | A100/H100/4090 原生支援，2026 推薦 |

**結論**：bf16 保留了 fp32 的指數位數（相同動態範圍），幾乎消除了 fp16 的溢位問題，
是 2026 年大模型訓練的預設精度。

### `device_map='auto'` 的語意

- HuggingFace `accelerate` 會自動把模型層分配到可用裝置
- 優先順序：GPU → CPU RAM → 磁碟（disk offload）
- 無需手動呼叫 `.cuda()` 或指定 `device=0`
- 多 GPU 環境自動做 tensor parallelism

### `use_safetensors=True` vs pickle

| | safetensors | pickle (.bin) |
|--|-------------|---------------|
| 安全性 | 無程式碼執行 | 載入即執行 Python 任意程式碼 |
| 載入速度 | mmap 零拷貝，快 2-5x | 需完整反序列化 |
| 2026 Hub 預設 | 是 | 舊模型 fallback |

In [ ]:
# 2026 統一載入慣例：
#   - torch_dtype=torch.bfloat16  bf16：保留 fp32 動態範圍，比 fp16 更穩定
#   - device_map='auto'           自動分配到 GPU/CPU/disk，無需手動 .cuda()
#   - use_safetensors=True        優先使用 safetensors 格式（更安全、更快）
#   - low_cpu_mem_usage 已被 device_map='auto' 隱含，不需額外指定

# VRAM 需求：ChatGLM3-6B bf16 全精度約 14 GB
# VRAM 不足時：改用 ../01-4bits_training/chatglm3_lora_4bit.ipynb（4-bit 約 6-8 GB）
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",           # 自動分配到 GPU/CPU/disk
    torch_dtype=torch.bfloat16, # bf16：保留 fp32 動態範圍，比 fp16 更穩定
    use_safetensors=True,       # 優先使用 safetensors 格式（更安全、更快）
)
print(model.dtype)
print(model.hf_device_map)  # 顯示各層分配到的裝置

In [ ]:
# 確認各層參數名稱（用於決定 LoRA target_modules）
for name, param in model.named_parameters():
    print(name, param.shape, param.dtype)

## Step 6：LoRA 設定

### LoRA 超參數說明

| 參數 | 建議值 | 說明 |
|------|--------|------|
| `r` | 8-16 | 低秩分解的秩；越大表達力越強，VRAM 也越多 |
| `lora_alpha` | 32 | 縮放係數；等效學習率 = lr × alpha/r，通常設 2-4 倍的 r |
| `lora_dropout` | 0.1 | 防止 adapter 過擬合 |
| `target_modules` | `["query_key_value"]` | ChatGLM3 的 QKV 合併投影層 |
| `bias` | `"none"` | 不訓練 bias，節省記憶體 |
| `task_type` | `CAUSAL_LM` | 明確指定任務類型，影響 forward 邏輯 |

### 為何只對 `query_key_value` 加 LoRA？

ChatGLM3 的 attention 層把 Q/K/V 三個投影合併成一個 `query_key_value` 矩陣。
選擇這個層作為 LoRA target 可以以最小代價覆蓋最關鍵的注意力機制。
若要更強效果，可加入 `dense`（output projection）或 MLP 的 `dense_h_to_4h` 等層。

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

lora_config = LoraConfig(
    r=8,                          # 低秩矩陣的秩
    lora_alpha=32,                # 縮放係數，等效 lr scaling = alpha / r = 4
    target_modules=["query_key_value"],  # ChatGLM3 的 QKV 合併投影層
    lora_dropout=0.1,             # 防止過擬合
    bias="none",                  # 不訓練 bias（節省記憶體）
    task_type=TaskType.CAUSAL_LM, # 因果語言模型
)
print(lora_config)

In [ ]:
# 套用 LoRA：只有 adapter 權重可訓練，原始模型權重凍結
model = get_peft_model(model, lora_config)

# 確認可訓練參數比例（通常 < 1%）
model.print_trainable_parameters()

In [ ]:
# 確認哪些層可訓練（名稱含 'lora_' 的才是 adapter 權重）
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

## Step 7：訓練

### 方案 A（推薦）：SFTTrainer — 自動處理標籤遮罩

`trl.SFTTrainer` 是 2026 指令微調的標準路徑：
- 自動套用 `apply_chat_template()` 產生格式化字串
- 自動計算 response-only 的 -100 遮罩（`DataCollatorForCompletionOnlyLM` 機制）
- 自動 sequence packing（提升 GPU 利用率）
- 與 PEFT 的 `peft_config` 原生整合

### 方案 B（教學參考）：手刻 Trainer — 如 Step 4 的 `process_func` 所示

### TrainingArguments 現代化必填項目

| 新增項目 | 原因 |
|----------|------|
| `bf16=True` | 對應模型載入精度，混合精度訓練 |
| `warmup_ratio=0.1` | 避免訓練初期 lr 過大導致不穩定 |
| `lr_scheduler_type='cosine'` | cosine 衰減收斂更平滑 |
| `max_grad_norm=1.0` | 梯度裁剪，防止梯度爆炸 |
| `save_safetensors=True` | 儲存為安全格式 |
| `optim='adamw_torch_fused'` | Fused AdamW：比原版快 20-30%，且有 weight decay 修正（AdamW 優於 Adam）|
| `seed=42` | 可重現性 |

In [ ]:
# 方案 A：SFTTrainer（推薦路徑）
# formatting_func 只負責組成 chat messages，SFTTrainer 內部自動套 apply_chat_template

def formatting_func(example):
    """把原始資料欄位轉成 messages list，SFTTrainer 會自動套 apply_chat_template。"""
    instruction_text = "\n".join(
        [example["instruction"], example["input"]]
    ).strip()
    messages = [
        {"role": "user", "content": instruction_text},
        {"role": "assistant", "content": example["output"]},
    ]
    # tokenize=False: 回傳純文字，SFTTrainer 負責 tokenize
    return tokenizer.apply_chat_template(messages, tokenize=False)


sft_config = SFTConfig(
    output_dir="./chatglm3-lora-sft",
    # --- 批次與步數 ---
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 2 × 16 = 32
    num_train_epochs=1,
    max_steps=-1,                      # -1 = 用 num_train_epochs 決定
    # --- 學習率排程 ---
    learning_rate=1e-4,
    warmup_ratio=0.1,                  # 前 10% steps 做 warmup
    lr_scheduler_type="cosine",        # cosine 衰減
    max_grad_norm=1.0,                 # 梯度裁剪
    # --- 精度與最佳化 ---
    bf16=True,                         # 對應模型 bfloat16 載入精度
    optim="adamw_torch_fused",         # Fused AdamW（速度 +20-30%）
    # --- 序列長度 ---
    max_seq_length=256,
    # --- 日誌與儲存 ---
    logging_steps=10,
    save_steps=100,
    save_safetensors=True,             # 儲存為 safetensors 格式
    # --- 可重現性 ---
    seed=42,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds.select(range(6000)),  # 前 6000 筆，與原版一致
    formatting_func=formatting_func,
    processing_class=tokenizer,            # 2026 寫法：processing_class 取代 tokenizer 參數
)

print("Effective batch size:",
      sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)

## Step 8：執行訓練

In [ ]:
trainer.train()

## Step 9：儲存 LoRA Adapter

儲存時只需存 LoRA adapter 的差值權重（幾十 MB），不需存完整模型（幾 GB）。
推論時再合併：`PeftModel.from_pretrained(base_model, adapter_path)`。

In [ ]:
# 儲存 LoRA adapter（只存差值，約幾十 MB）
# safe_serialization=True：使用 safetensors 格式
model.save_pretrained("./chatglm3-lora-adapter", safe_serialization=True)
tokenizer.save_pretrained("./chatglm3-lora-adapter")
print("Adapter saved to ./chatglm3-lora-adapter")

## Step 10：模型推理

推論使用 `apply_chat_template()` + `model.generate()`，與訓練側路徑一致：
- 訓練用 `apply_chat_template(messages, tokenize=False, add_generation_prompt=True)`
- 推論同一套，確保訓練/推論 distribution 一致
- 可移植到任何實作了 chat template 的模型（Llama 3、Qwen 2.5 等）

In [ ]:
model.eval()

# 組 prompt（與訓練時的 formatting_func 路徑一致）
messages = [
    {"role": "user", "content": "數學考試怎麼考高分？"},
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # 加上 <|assistant|>\n，告知模型開始生成
)

# tokenize（在模型所在裝置上）
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,  # 避免警告
    )

# 只解碼新生成的 token（去掉 prompt 部分）
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print("問：數學考試怎麼考高分？")
print("答：", response)

In [ ]:
# 批次推論示範（同一 API，只改 messages 內容）
test_questions = [
    "如何提升英語閱讀能力？",
    "Python 和 Java 哪個更適合初學者？",
]

for question in test_questions:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,   # greedy decoding，確定性輸出
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    print(f"問：{question}")
    print(f"答：{response}")
    print("-" * 60)

## 小結

本 notebook 示範了 2026 年 LoRA 微調的完整流程，涵蓋以下核心實踐：

- **資料載入**：使用 HF Hub dataset id，做到零硬路徑依賴
- **模型載入**：`torch_dtype=torch.bfloat16` + `device_map='auto'` + `use_safetensors=True` 三件套
- **Chat prompt**：`apply_chat_template()` 統一訓練與推論路徑，跨模型可攜
- **標籤遮罩**：SFTTrainer 自動處理 response-only 的 -100 遮罩
- **訓練器**：`SFTTrainer` + `SFTConfig` 是指令微調的標準路徑
- **最佳化器**：`adamw_torch_fused` 提供 weight decay 修正與速度提升
- **儲存格式**：`safe_serialization=True` 輸出 safetensors

## 練習

1. **調整 LoRA rank**：分別用 `r=4`、`r=16`、`r=32` 訓練，比較可訓練參數比例與最終 loss 的變化。`lora_alpha` 應如何相應調整？

2. **擴展 target_modules**：在 `query_key_value` 之外加入 `dense` 層，觀察 `print_trainable_parameters()` 的輸出變化，以及訓練速度的影響。

3. **理解 -100 遮罩**：在 `process_func` 中，把 `labels` 改為不設 -100（即 `labels = input_ids`），重新訓練後觀察模型推論行為有何不同？為什麼？

4. **apply_chat_template 可攜性驗證**：把 `MODEL_ID` 換成 `Qwen/Qwen2.5-7B-Instruct`，只改 model id，確認 `apply_chat_template()` + `SFTTrainer` 的其餘程式碼不需改動即可執行。

5. **推論參數實驗**：在 Step 10 的 `model.generate()` 中，比較 `do_sample=True` 的 temperature=0.3 / 0.7 / 1.2，以及 `do_sample=False`（greedy），觀察回答品質與多樣性的差異。